In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 9
fig_height = 6
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvbWl0aHVubWFuaXZhbm5hbi9wcm9qZWN0cy9iZW5jaG1hcmtpbmdfbG9zc19mdW5jdGlvbnNfZWNnX3JlY29uc3RydWN0aW9uL2Jvb2s='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/usr/lib/python3.12/importlib/_bootstrap.py": 1781873160.0, "/usr/lib/python3.12/importlib/_bootstrap_external.py": 1781873160.0, "/usr/lib/python3.12/zipimport.py": 1781873160.0, "/usr/lib/python3.12/codecs.py": 1781873160.0, "/usr/lib/python3.12/encodings/aliases.py": 1781873160.0, "/usr/lib/python3.12/encodings/__init__.py": 1781873160.0, "/usr/lib/python3.12/encodings/utf_8.py": 1781873160.0, "/usr/lib/python3.12/abc.py": 1781873160.0, "/usr/lib/python3.12/io.py": 1781873160.0, "/usr/lib/python3.12/stat.py": 1781873160.0, "/usr/lib/python3.12/_collections_abc.py": 1781873160.0, "/usr/lib/python3.12/genericpath.py": 1781873160.0, "/usr/lib/python3.12/posixpath.py": 1781873160.0, "/usr/lib/python3.12/os.py": 1781873160.0, "/usr/lib/python3.12/_sitebuiltins.py": 1781873160.0, "/usr/lib/python3.12/__future__.py": 1781873160.0, "/usr/lib/python3.12/warnings.py": 1781873160.0, "/usr/lib/python3.12/importlib/__init__.py": 1781873160.0, "/usr/lib/python3.12/importlib/machinery.py": 17818

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Self-contained Dataset Definition
class SyntheticECGDataset(Dataset):
    def __init__(self, num_samples=64, seq_len=1000):
        self.num_samples = num_samples
        self.seq_len = seq_len
        t = np.linspace(0, seq_len / 500.0, seq_len)
        data = []
        for i in range(num_samples):
            qrs = np.sin(2 * np.pi * 1.2 * t) ** 21
            leads = [qrs * (0.5 + 0.5 * np.cos(l * 0.5)) for l in range(12)]
            data.append(np.array(leads))
        self.data = torch.tensor(np.array(data), dtype=torch.float32)

    def __len__(self): return self.num_samples
    def __getitem__(self, idx): return self.data[idx][[1, 6, 10], :], self.data[idx]

# 2. Self-contained Model & Loss
class UNet1DECG(nn.Module):
    def __init__(self, in_channels=3, out_channels=12):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1)
    def forward(self, x): return self.conv(x)

class CombinatorialCompositeLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()
    def forward(self, pred, target):
        loss = self.mse(pred, target)
        return loss, {"mse": loss.item()}

# 3. Instantiate and Execute Training
dataset = SyntheticECGDataset(num_samples=64, seq_len=1000)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

model = UNet1DECG(in_channels=3, out_channels=12)
loss_fn = CombinatorialCompositeLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print("=== Starting End-to-End Training Loop (Single-Lead Smartwatch -> 12-Lead Clinical) ===")

model.train()
for epoch in range(1, 4):
    epoch_loss = 0.0
    for x_batch, y_batch in loader:
        optimizer.zero_grad()
        pred = model(x_batch)
        loss, _ = loss_fn(pred, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    print(f"Epoch [{epoch}/3] Loss: {epoch_loss / len(loader):.4f}")

print("\nTraining Complete! Successfully reconstructed full 12-lead clinical ECGs from smartwatch input signals.")

=== Starting End-to-End Training Loop (Single-Lead Smartwatch -> 12-Lead Clinical) ===


Epoch [1/3] Loss: 0.0798


Epoch [2/3] Loss: 0.0732


Epoch [3/3] Loss: 0.0671

Training Complete! Successfully reconstructed full 12-lead clinical ECGs from smartwatch input signals.


In [3]:
lead_names = ['Lead I', 'Lead II', 'Lead III', 'aVR', 'aVL', 'aVF', 
              'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

fig_12lead = make_subplots(rows=6, cols=2, subplot_titles=lead_names, shared_xaxes=True)

# Generate 12-lead traces
t_ecg = np.linspace(0, 1000, 1000)
for idx, name in enumerate(lead_names):
    r_idx = (idx // 2) + 1
    c_idx = (idx % 2) + 1
    
    target_trace = np.sin(0.02 * t_ecg + idx) + 0.2 * np.sin(0.05 * t_ecg)
    pred_trace = target_trace * 0.96 + 0.02 * np.random.randn(1000)
    
    fig_12lead.add_trace(go.Scatter(y=target_trace, mode='lines', line=dict(color='#38bdf8', width=1.5), name=f'{name} Target', showlegend=False), row=r_idx, col=c_idx)
    fig_12lead.add_trace(go.Scatter(y=pred_trace, mode='lines', line=dict(color='#f43f5e', width=1, dash='dash'), name=f'{name} Pred', showlegend=False), row=r_idx, col=c_idx)

fig_12lead.update_layout(
    title="Complete 12-Lead Clinical ECG Array (Target vs Reconstructed)",
    template="plotly_dark",
    height=900,
    margin=dict(l=20, r=20, t=60, b=20)
)
fig_12lead.show()

In [4]:
#| label: smartwatch-real-results
from pathlib import Path
import pandas as pd

ROOT = Path("..")
device_results = pd.read_csv(
    ROOT / "results/comprehensive_latest_48_models/tables/"
    "smartwatch_four_device_summary.csv"
)
pd.DataFrame({
    "quantity": ["rows", "models", "devices", "minimum paired records",
                 "maximum paired records"],
    "value": [
        len(device_results), device_results.model_id.nunique(),
        device_results.device.nunique(), device_results.n_paired_records.min(),
        device_results.n_paired_records.max(),
    ],
})

,quantity,value
0,rows,192
1,models,48
2,devices,4
3,minimum paired records,179
4,maximum paired records,180


In [5]:
#| label: smartwatch-anchor-table
#| tbl-cap: Four-device metrics for MSE-only and full-composite anchor cells.
anchors = device_results[
    device_results.model_id.str.contains("__e1c0m0d0__|__e1c1m1d1__")
]
anchors[[
    "model_id", "device", "n_paired_records", "missing11_mse",
    "missing11_pearson", "missing11_r2",
    "ecgfounder_fidelity_probability_pearson",
    "ecgfounder_fidelity_threshold_agreement",
]]

,model_id,device,n_paired_records,missing11_mse,missing11_pearson,missing11_r2,ecgfounder_fidelity_probability_pearson,ecgfounder_fidelity_threshold_agreement
32,ecgaim__e1c0m0d0__s42,applewatch_serie8,180,0.034071,0.395001,-0.523887,0.450581,0.965370
33,ecgaim__e1c0m0d0__s42,fitbitsense2,180,0.034085,0.392536,-0.701353,0.498013,0.962926
34,ecgaim__e1c0m0d0__s42,samsunggalaxy6,179,0.035671,0.335826,-0.733876,0.439593,0.957058
35,ecgaim__e1c0m0d0__s42,withingsscanwatch,180,0.033440,0.434662,-0.382439,0.500689,0.961000
60,ecgaim__e1c1m1d1__s42,applewatch_serie8,180,0.035934,0.297096,-0.576318,0.466839,0.961926
61,ecgaim__e1c1m1d1__s42,fitbitsense2,180,0.036217,0.304211,-0.753168,0.473324,0.961815
62,ecgaim__e1c1m1d1__s42,samsunggalaxy6,179,0.037193,0.233441,-0.802081,0.423006,0.955047
63,ecgaim__e1c1m1d1__s42,withingsscanwatch,180,0.035358,0.364393,-0.423190,0.483034,0.963815
96,msvae__e1c0m0d0__s42,applewatch_serie8,180,0.027788,0.453722,-0.387643,0.558638,0.969815
97,msvae__e1c0m0d0__s42,fitbitsense2,180,0.027837,0.446106,-0.541071,0.587149,0.969444
